# 06｜月份效應：一月效應與五月賣股策略
**理論來源：Wachtel (1942)、Rozeff & Kinney (1976) — 一月效應**
**Bouman & Jacobsen (2002) — Sell in May and Go Away**

> **核心問題：股市在某些月份真的比較容易漲嗎？**
>
> 一月效應是最早被發現的市場異常：每年一月的報酬率系統性偏高。
> 「五月賣股走人」則說：每年五月到十月股市表現差，要待在現金。
>
> 但自從被學術界公開後，這些效應還成立嗎？
> 本 Notebook 用 145 年資料驗證，並追蹤效應的「消亡過程」——
> 展示**市場效率自我修正**的力量。

## 「五月賣股，十月回來」——你信嗎？

台灣版本的說法更豐富：

- *「農曆年前主力作帳拉尾盤，年後一定反彈」*
- *「七月八月是傳統淡季，電子股都不好」*
- *「Q4 消費旺季，零售股年底必漲」*

這些說法在股市論壇、LINE 群組裡流傳了幾十年。某些年份好像還真的準。

**但問題是：如果這些規律真的那麼穩定，為什麼不是每個人都靠它發財？**

這一章，我們用 Fama-French 美股月度資料（1926 年至今，將近百年），
用統計方法正式測試「月份效應」是否真實存在——以及就算存在，套利之後還剩多少。

## 🎯 學習目標
1. 用統計檢定驗證「一月效應」是否在歷史數據中顯著存在
2. 回測「五月賣股（Sell in May）」策略的實際績效
3. 了解「市場異象被發現後會消失」的一般規律

## 匯入套件

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

matplotlib.rcParams['font.family'] = ['Microsoft YaHei', 'SimHei', 'Heiti TC', 'PingFang HK', 'STHeiti', 'Arial Unicode MS', 'sans-serif']
matplotlib.rcParams['axes.unicode_minus'] = False
print("✅ 匯入完成")

## 一、載入資料

In [ ]:
import os

df = pd.read_csv("data/shiller_data.csv", parse_dates=['date'], index_col='date')
df['div_yield_m'] = df['dividend'] / df['price'] / 12
df['ret']         = df['real_price'].pct_change() + df['div_yield_m'].shift(1)
df = df.dropna(subset=['ret']).copy()
df['month'] = df.index.month
df['year']  = df.index.year

MONTHS = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
print(f"資料期間：{df.index[0].year}–{df.index[-1].year}，共 {len(df)} 個月")

## 二、各月份平均報酬

In [ ]:
monthly = df.groupby('month')['ret'].agg(['mean','std','count'])
monthly['mean_pct'] = monthly['mean'] * 100
monthly['se_pct']   = monthly['std'] / np.sqrt(monthly['count']) * 100

# 配色：一月紅、夏季橘、冬季綠
colors = ['#e74c3c' if m == 1 else
          '#f39c12' if 5 <= m <= 10 else
          '#2ecc71'
          for m in range(1, 13)]

fig, ax = plt.subplots(figsize=(11, 5))
bars = ax.bar(range(1, 13), monthly['mean_pct'], color=colors, alpha=0.85, edgecolor='white')
ax.errorbar(range(1, 13), monthly['mean_pct'],
            yerr=monthly['se_pct'] * 1.96,
            fmt='none', color='black', capsize=4, linewidth=1)
ax.axhline(df['ret'].mean() * 100, color='navy', linestyle='--', linewidth=1.2,
           label=f"全期月均 {df['ret'].mean()*100:.3f}%")
ax.axhline(0, color='black', linewidth=0.5)
ax.set_xticks(range(1, 13)); ax.set_xticklabels(MONTHS)
ax.set_ylabel('平均月報酬（%）')
ax.set_title(f'S&P 500 各月份平均實質報酬（{df.index[0].year}–{df.index[-1].year}）', fontsize=12)

legend_handles = [
    mpatches.Patch(color='#e74c3c', alpha=0.85, label='一月（January Effect）'),
    mpatches.Patch(color='#f39c12', alpha=0.85, label='五到十月（Sell in May 空窗期）'),
    mpatches.Patch(color='#2ecc71', alpha=0.85, label='十一到四月（冬季強勢期）'),
]
ax.legend(handles=legend_handles, fontsize=9)

for bar, val in zip(bars, monthly['mean_pct']):
    offset = 0.03 if val >= 0 else -0.10
    ax.text(bar.get_x() + bar.get_width()/2, val + offset,
            f'{val:.2f}%', ha='center', va='bottom', fontsize=8)
plt.tight_layout(); plt.show()

jan = monthly.loc[1, 'mean_pct']
others = df[df['month'] != 1]['ret'].mean() * 100
print(f"一月均值：{jan:.3f}%")
print(f"其他月份均值：{others:.3f}%")
print(f"一月超額報酬：{jan - others:+.3f}%")

## 三、一月效應統計顯著性

In [ ]:
jan_rets   = df[df['month'] == 1]['ret'].values
other_rets = df[df['month'] != 1]['ret'].values
t_stat, p_val = stats.ttest_ind(jan_rets, other_rets)

print(f"一月（n={len(jan_rets)}）vs 其他月份（n={len(other_rets)}）")
print(f"t 統計量：{t_stat:.3f}")
print(f"p 值：{p_val:.3f}")
print()
if p_val < 0.05:
    print("✅ p < 0.05，一月效應在整體資料中統計上顯著")
elif p_val < 0.10:
    print("⚠️ 0.05 < p < 0.10，邊緣顯著")
else:
    print("❌ p ≥ 0.10，整體上一月效應不顯著")

print()
print("各月份與其他月份 t 檢定：")
for m in range(1, 13):
    m_ret  = df[df['month'] == m]['ret'].values
    ot_ret = df[df['month'] != m]['ret'].values
    t, p   = stats.ttest_ind(m_ret, ot_ret)
    sig    = ' ★' if p < 0.05 else (' △' if p < 0.10 else '')
    print(f"  {MONTHS[m-1]:>3}: 均值 {m_ret.mean()*100:+.3f}%  p={p:.3f}{sig}")

## 四、一月效應隨時代消亡了嗎？

效應被學術界公開後，投資人開始在一月前提前布局，套利使效應縮小。
我們把 145 年切成三段來觀察。

In [ ]:
eras = [
    ('早期（1881–1950）',   1881, 1950),
    ('發現期（1951–1990）', 1951, 1990),
    ('後公開（1991–今）',   1991, 2030),
]

fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)

for ax, (label, y0, y1) in zip(axes, eras):
    sub = df[(df['year'] >= y0) & (df['year'] <= y1)]
    avg = sub.groupby('month')['ret'].mean() * 100
    col = ['#e74c3c' if m == 1 else '#bdc3c7' for m in range(1, 13)]
    ax.bar(range(1, 13), avg, color=col, alpha=0.85, edgecolor='white')
    ax.axhline(avg.mean(), color='navy', linestyle='--', linewidth=1)
    ax.axhline(0, color='black', linewidth=0.5)
    ax.set_xticks(range(1, 13))
    ax.set_xticklabels(MONTHS, rotation=45, ha='right', fontsize=8)
    n_years = len(sub['year'].unique())
    ax.set_title(f'{label}\n（{n_years} 年）\n一月：{avg.iloc[0]:+.2f}%', fontsize=10)
    if y0 == 1881:
        ax.set_ylabel('平均月報酬（%）')

plt.suptitle('一月效應的消亡過程（被研究後逐漸縮小）', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

## 五、Sell in May and Go Away — 冬季 vs 夏季

In [ ]:
df['season'] = df['month'].apply(lambda m: 'summer' if 5 <= m <= 10 else 'winter')

summer = df[df['season'] == 'summer']['ret']
winter = df[df['season'] == 'winter']['ret']

summer_ann = ((1 + summer.mean()) ** 12 - 1) * 100
winter_ann = ((1 + winter.mean()) ** 12 - 1) * 100
t2, p2 = stats.ttest_ind(winter.values, summer.values)

print("季節性報酬比較（五到十月 vs 十一到四月）：")
print(f"  夏季（5–10月）月均：{summer.mean()*100:.3f}%  年化約 {summer_ann:.1f}%")
print(f"  冬季（11–4月）月均：{winter.mean()*100:.3f}%  年化約 {winter_ann:.1f}%")
print(f"  冬季溢酬：{(winter.mean()-summer.mean())*100:+.3f}% / 月")
print(f"  t={t2:.2f}, p={p2:.3f}  {'★ 顯著' if p2 < 0.05 else ''}")

# 滾動 20 年冬季溢酬
roll_prem, roll_year = [], []
for y0 in range(df['year'].min(), df['year'].max() - 20):
    sub = df[(df['year'] >= y0) & (df['year'] < y0 + 20)]
    w = sub[sub['season'] == 'winter']['ret'].mean()
    s = sub[sub['season'] == 'summer']['ret'].mean()
    roll_prem.append((w - s) * 100)
    roll_year.append(y0 + 10)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# 左：各月份，夏冬配色
avg_m = df.groupby('month')['ret'].mean() * 100
col2 = ['#f39c12' if 5 <= m <= 10 else '#2ecc71' for m in range(1, 13)]
axes[0].bar(range(1, 13), avg_m, color=col2, alpha=0.85, edgecolor='white')
axes[0].set_xticks(range(1, 13)); axes[0].set_xticklabels(MONTHS)
axes[0].axhline(0, color='black', linewidth=0.5)
axes[0].set_ylabel('平均月報酬（%）')
axes[0].set_title('各月份報酬\n（橘 = 夏季，綠 = 冬季）', fontsize=11)
from matplotlib.patches import Patch
axes[0].legend(handles=[
    Patch(color='#f39c12', alpha=0.85, label='夏季（5–10 月）'),
    Patch(color='#2ecc71', alpha=0.85, label='冬季（11–4 月）'),
])

# 右：滾動 20 年冬季溢酬
col3 = ['#2ecc71' if p > 0 else '#e74c3c' for p in roll_prem]
axes[1].bar(roll_year, roll_prem, color=col3, alpha=0.75, width=1.5)
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].set_xlabel('20 年窗口中點'); axes[1].set_ylabel('冬季 − 夏季 月均報酬（%）')
axes[1].set_title('冬季溢酬是否穩定？\n滾動 20 年窗口', fontsize=11)

plt.suptitle('Sell in May and Go Away 驗證', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

## 六、討論

**結論：**
- 一月效應在早期資料（1881–1950）相對明顯，但 **1991 年後大幅減弱至幾乎消失**
- 冬季溢酬（Sell in May）整體存在，但在某些 20 年窗口為負，策略表現**不穩定**
- 兩者都符合「被發現 → 被套利 → 消亡」的市場自我糾正過程

**這告訴我們什麼？**
> **市場異常一旦廣為人知，就會被套利消除。**
> 這是弱式效率市場假說（Weak-form EMH）的最佳示範。
> 但效應的消亡往往需要幾十年，在此期間仍有人獲利。

**思考問題：**
1. 一月效應最初為什麼存在？（提示：稅務損失收割、基金年末換倉、年初再配置）
2. 把交易成本和稅負算進去，「五月賣股走人」策略實際上還能賺嗎？
3. 台股有一月效應嗎？農曆年前後有異常報酬嗎？（文化因素可能產生不同的季節效應）
4. 如果你發現一個新的市場異常，你要怎麼確認它是真實的而不是資料探勘出來的巧合？

## 七、這跟你有什麼關係？

**不要跟隨「幾月買股必漲」的建議——即使效應真的存在，套利後你也拿不到。**

最直觀的驗證：如果你真的每年五月賣出、十月底買回（Sell in May），
和完全不動（Buy and Hold）相比，145 年下來結果怎樣？

In [ ]:
# 模擬 Sell in May 策略 vs Buy and Hold（1881–今）
p_bah, p_sim = 1.0, 1.0
bah_track, sim_track, date_track = [1.0], [1.0], [df.index[0]]

for date, row in df.iterrows():
    p_bah *= (1 + row['ret'])
    if row['month'] not in [5, 6, 7, 8, 9, 10]:
        p_sim *= (1 + row['ret'])   # 冬季持有
    # 夏季現金，假設利率 0（對 Sell-in-May 策略有利的保守假設）
    bah_track.append(p_bah)
    sim_track.append(p_sim)
    date_track.append(date)

n_years = len(df) / 12
bah_ann = (p_bah ** (1 / n_years) - 1) * 100
sim_ann = (p_sim ** (1 / n_years) - 1) * 100

print(f"Buy and Hold：年化 {bah_ann:.2f}%，最終資產 {p_bah:,.0f}x")
print(f"Sell in May ：年化 {sim_ann:.2f}%，最終資產 {p_sim:,.0f}x")
print(f"每年差距：{bah_ann - sim_ann:.2f}%（Buy and Hold 勝）")
print()
print(f"這還沒算：每年兩次的交易成本 + 潛在資本利得稅")
print(f"結論：跟隨季節性操作建議，不但沒有幫助，還會讓你少賺 {bah_ann - sim_ann:.2f}%/年")

fig, ax = plt.subplots(figsize=(12, 4))
ax.semilogy(date_track, bah_track, label=f'Buy and Hold（{bah_ann:.1f}%/年）',
            color='#2ecc71', linewidth=1)
ax.semilogy(date_track, sim_track, label=f'Sell in May（{sim_ann:.1f}%/年）',
            color='#e74c3c', linewidth=1, linestyle='--')
ax.set_ylabel('累積資產（對數）')
ax.set_title('Buy and Hold vs Sell in May（1881–今）', fontsize=12)
ax.legend(fontsize=10)
plt.tight_layout(); plt.show()

## 📌 本章重點摘要
| 概念 | 結論 |
|------|------|
| 一月效應 | 統計顯著，但近年效果縮小 |
| Sell in May | 歷史上平均錯過報酬；5–10月不一定是壞月份 |
| 被發現後消失 | 學術發表 → 套利 → 效果消退（McLean & Pontiff）|
| 個人應用 | 日曆策略不可靠；紀律持有優於時機選擇 |

> **下一章：** 因子動物園——400 個學術因子，哪些是真的？